# Predictions for data-test

In [13]:
import json

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

data = pd.read_excel("../data-train.xlsx")
data_test = pd.read_excel("../data-test.xlsx")

FIT_ON_ALL_GAMES = False

## p_challenge

In [14]:
metadata = ["play_id", "venue_id", "pitcher_id", "batter_id", "catcher_id", "umpire_id", "game_id"]
non_target_features = ["challenge_source", "is_success"]
X_c = data.drop(metadata + non_target_features, axis=1).dropna()
y_c = X_c.pop("is_challenge")
groups_c = data.loc[X_c.index, "game_id"]

positive_values_c = {
    "is_top_inning": True,
    "pitcher_hand": "Right",
    "batter_hand": "Right",
    "pitch_result": "called_strike",
}
for f, positive in positive_values_c.items():
    X_c[f] = (X_c[f] == positive).astype("int64")

In [15]:

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=67).split(X_c, y_c, groups_c))
fit_idx = np.arange(len(X_c)) if FIT_ON_ALL_GAMES else train_idx
model_c = XGBClassifier(**json.load(open("xgb_params_p_challenge.json"))).fit(X_c.iloc[fit_idx], y_c.iloc[fit_idx])
fit_rows_c = X_c.index[fit_idx]  # rows the model was fit on

if not FIT_ON_ALL_GAMES:  # should match XGBoost (tuned) in p_challenge.ipynb
    p = model_c.predict_proba(X_c.iloc[test_idx])[:, 1]
    print("held-out log loss:", round(log_loss(y_c.iloc[test_idx], p), 4))

held-out log loss: 0.0791


## p_success_g_challenge

In [ ]:
# same preprocessing as p_success_g_challenge.ipynb
given_challenge = data[data["is_challenge"] == 1]
ignore = given_challenge.columns.difference(non_target_features)
given_challenge = given_challenge.dropna(subset=ignore)
given_challenge = given_challenge.drop(columns=["is_challenge"])
groups_s = given_challenge["game_id"]
given_challenge = given_challenge.drop(metadata, axis=1)

positive_values_s = {
    "is_top_inning": True,
    "pitcher_hand": "Right",
    "batter_hand": "Right",
    "pitch_result": "called_strike",
    "challenge_source": "hitting_team",
    "is_success": True,
}
for f, positive in positive_values_s.items():
    given_challenge[f] = (given_challenge[f] == positive).astype("int64")

X_s = given_challenge.drop(["is_success"], axis=1)
y_s = given_challenge["is_success"]

In [17]:
# tuned XGBoost from p_success_g_challenge.ipynb, on the same split and seed
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=67).split(X_s, y_s, groups_s))
fit_idx = np.arange(len(X_s)) if FIT_ON_ALL_GAMES else train_idx
model_s = XGBClassifier(**json.load(open("xgb_params_p_success.json"))).fit(X_s.iloc[fit_idx], y_s.iloc[fit_idx])
fit_rows_s = X_s.index[fit_idx]  # rows the model was fit on

if not FIT_ON_ALL_GAMES:  # should match XGBoost (tuned) in p_success_g_challenge.ipynb
    p = model_s.predict_proba(X_s.iloc[test_idx])[:, 1]
    print("held-out log loss:", round(log_loss(y_s.iloc[test_idx], p), 4))

held-out log loss: 0.3237


# Filling data-test

In [18]:
# fill a copy of data-test; data_test stays exactly as read from data-test.xlsx
test_out = data_test.copy()

# we assume only called strikes should only be challenged by hitters, called balls by pitching team
test_out["challenge_source"] = np.where(data_test["pitch_result"] == "called_strike", "hitting_team", "pitching_team")

# p_challenge
# rows missing any input features are left as NaN
X_new = data_test[X_c.columns].copy()
complete_c = X_new.notna().all(axis=1)
for f, positive in positive_values_c.items():
    X_new[f] = (X_new[f] == positive).astype("int64")
test_out["p_challenge"] = np.nan
test_out.loc[complete_c, "p_challenge"] = model_c.predict_proba(X_new[complete_c])[:, 1]

# p_success_g_challenge
# every pitch is scored as if it were challenged (is_challenge = 1),
# with the challenge source set above
# rows missing any input are left as NaN
X_new = test_out.assign(is_challenge=1)[X_s.columns].copy()
complete_s = X_new.notna().all(axis=1)
for f, positive in positive_values_s.items():
    if f in X_new.columns:  # is_success is the label, not an input
        X_new[f] = (X_new[f] == positive).astype("int64")
test_out["p_success_g_challenge"] = np.nan
test_out.loc[complete_s, "p_success_g_challenge"] = model_s.predict_proba(X_new[complete_s])[:, 1]

print(f"{len(test_out)} rows | left as NaN (missing inputs): p_challenge {(~complete_c).sum()}, "
      f"p_success_g_challenge {(~complete_s).sum()}")
# data_test[data_test["is_challenge"]==True][["p_challenge", "p_success_g_challenge"]].describe()
assert list(test_out.columns) == list(data_test.columns)  # every original column, in the original order
test_out.head()

34972 rows | left as NaN (missing inputs): p_challenge 5, p_success_g_challenge 5


,play_id,venue_id,balls,strikes,outs,inning,is_top_inning,pitcher_id,pitcher_hand,batter_id,...,plate_x,plate_z,sz_top,sz_bot,pa_of_game,pitch_of_pa,challenge_source,game_id,p_challenge,p_success_g_challenge
0,F7J7VL1nVLMUBJdq,82346,0,0,2,6,False,AOFhIJmN,Right,i4mn3ged,...,1.132149,3.282238,3.189,1.610,50,1,pitching_team,53715308,0.000579,0.195651
1,K1IFMrPYKocUXGZe,901248,0,0,1,8,False,AOFhIJmN,Right,73k7IVed,...,0.754192,2.754101,3.165,1.597,64,1,hitting_team,82918948,0.033477,0.060105
2,bK9s0mbrbgzvc66y,901248,0,1,1,8,False,AOFhIJmN,Right,73k7IVed,...,-0.992932,2.603603,3.165,1.597,64,2,pitching_team,82918948,0.041046,0.016298
3,pk6ELmdkDec9yYDK,82346,1,2,1,7,False,AOFhIJmN,Right,jkmUP0RQ,...,0.969064,2.231950,3.035,1.532,58,5,hitting_team,53715308,0.396900,0.873793
4,vcFwWo2TULtFHJAq,82346,0,1,0,5,False,AOFhIJmN,Right,jkmUP0RQ,...,-0.852970,1.242416,3.035,1.532,40,2,pitching_team,53715308,0.000548,0.134263


In [19]:
# export a new file; data-test.xlsx is never overwritten
test_out.to_excel("data-test-predictions.xlsx", index=False)
test_out[["play_id", "pitch_result", "challenge_source", "p_challenge", "p_success_g_challenge"]].head()

,play_id,pitch_result,challenge_source,p_challenge,p_success_g_challenge
0,F7J7VL1nVLMUBJdq,ball,pitching_team,0.000579,0.195651
1,K1IFMrPYKocUXGZe,called_strike,hitting_team,0.033477,0.060105
2,bK9s0mbrbgzvc66y,ball,pitching_team,0.041046,0.016298
3,pk6ELmdkDec9yYDK,called_strike,hitting_team,0.396900,0.873793
4,vcFwWo2TULtFHJAq,ball,pitching_team,0.000548,0.134263


In [20]:
# the same predictions for every training pitch, added to a copy of data-train. every original column is kept
# as recorded (including challenge_source, is_challenge and is_success), so the rule-based challenge source
# goes in a new column instead of replacing the recorded one
train_out = data.copy()
train_out["challenge_source_pred"] = np.where(data["pitch_result"] == "called_strike", "hitting_team", "pitching_team")

# p_challenge
X_new = data[X_c.columns].copy()
complete_c = X_new.notna().all(axis=1)
for f, positive in positive_values_c.items():
    X_new[f] = (X_new[f] == positive).astype("int64")
train_out["p_challenge"] = np.nan
train_out.loc[complete_c, "p_challenge"] = model_c.predict_proba(X_new[complete_c])[:, 1]

# p_success_g_challenge: scored as if challenged, with the rule-based challenge source (as for data-test)
X_new = data.assign(is_challenge=1, challenge_source=train_out["challenge_source_pred"])[X_s.columns].copy()
complete_s = X_new.notna().all(axis=1)
for f, positive in positive_values_s.items():
    if f in X_new.columns:
        X_new[f] = (X_new[f] == positive).astype("int64")
train_out["p_success_g_challenge"] = np.nan
train_out.loc[complete_s, "p_success_g_challenge"] = model_s.predict_proba(X_new[complete_s])[:, 1]

# rows each model was fit on: predictions there are in-sample, so they look better than they would on new games
train_out["p_challenge_in_sample"] = data.index.isin(fit_rows_c)
train_out["p_success_in_sample"] = data.index.isin(fit_rows_s)

print(f"{len(train_out)} rows | left as NaN (missing inputs): p_challenge {(~complete_c).sum()}, "
      f"p_success_g_challenge {(~complete_s).sum()}")
assert list(train_out.columns[:len(data.columns)]) == list(data.columns)  # original columns first, unchanged
train_out[["p_challenge", "p_success_g_challenge"]].describe()

309063 rows | left as NaN (missing inputs): p_challenge 440, p_success_g_challenge 440


,p_challenge,p_success_g_challenge
count,3.086230e+05,308623.000000
mean,2.707046e-02,0.134840
std,6.396970e-02,0.221797
min,1.202407e-07,0.000547
25%,5.904210e-05,0.022074
50%,2.633342e-03,0.049914
75%,2.082382e-02,0.116287
max,9.134545e-01,0.998459


In [21]:
# export a new file; data-train.xlsx is never overwritten (writing ~309,000 rows takes a few minutes)
train_out.to_excel("../p2/data-train-predictions.xlsx", index=False)
train_out[["play_id", "pitch_result", "challenge_source", "challenge_source_pred", "p_challenge", "p_success_g_challenge"]].head()

,play_id,pitch_result,challenge_source,challenge_source_pred,p_challenge,p_success_g_challenge
0,c0UdVTCjQ5SUsJ89,ball,NaN,pitching_team,0.000066,0.026283
1,h8pmSGX80G2OMWkR,called_strike,NaN,hitting_team,0.343817,0.590569
2,6n0nKEkw0o3e2Hyp,ball,NaN,pitching_team,0.122334,0.736176
3,aAgwjqGrKTKOjn1f,ball,NaN,pitching_team,0.001227,0.078011
4,mBsYKFCsq0hvdRba,ball,NaN,pitching_team,0.000071,0.027597
